# Add: Embeddings, and Linear Output Layer

In [31]:
data = [
  "She composes songs and practices piano daily.",
  "He reads books and explores the nearby caves.",
  "He reads novel and climbs mountains every weekend.",
  "She composes songs and writes novels.",
  "He reads newspaper and solves complex puzzles.",
  "She composes music and organizes exhibitions regularly.",
  "He reads books and builds small wooden models.",
  "He reads books and participates in local science fairs.",
  "She composes songs and curates art projects.",
  "She composes tunes and designs jewelry for her friends.",
  "He reads everyday and documents wildlife photography trips.",
  "She composes harmonies and experiments with digital music.",
  "He reads novel and trains for local marathons.",
  "She composes soundtracks and collaborates with creative filmmakers.",
  "He reads newspaper and studies navigation using maps and stars.",
  "She composes rhythms and teaches music."
]

In [32]:
VOCAB_SIZE = 70
CONTEXT_LEN = 6
EMB_DIM = 24
BATCH_SIZE = 4
EPOCHS = 100

In [33]:
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

In [34]:
_ = torch.manual_seed(123)

In [35]:
def separate_dots(s):
    s = re.sub(r"\.", " . ", s)
    return s.strip()

text_data = [separate_dots(x) for x in data]

In [36]:
example_data = text_data[0]
print(example_data)

She composes songs and practices piano daily .


In [37]:
vocab = set()

for text in text_data:
    for word in text.split():
        vocab.add(word)

vocab = sorted(vocab)

print(len(vocab))

70


In [38]:
# Display token:word mappings
for i, word in enumerate(vocab):
    print(f"{i}: {word}")

0: .
1: He
2: She
3: and
4: art
5: books
6: builds
7: caves
8: climbs
9: collaborates
10: complex
11: composes
12: creative
13: curates
14: daily
15: designs
16: digital
17: documents
18: every
19: everyday
20: exhibitions
21: experiments
22: explores
23: fairs
24: filmmakers
25: for
26: friends
27: harmonies
28: her
29: in
30: jewelry
31: local
32: maps
33: marathons
34: models
35: mountains
36: music
37: navigation
38: nearby
39: newspaper
40: novel
41: novels
42: organizes
43: participates
44: photography
45: piano
46: practices
47: projects
48: puzzles
49: reads
50: regularly
51: rhythms
52: science
53: small
54: solves
55: songs
56: soundtracks
57: stars
58: studies
59: teaches
60: the
61: trains
62: trips
63: tunes
64: using
65: weekend
66: wildlife
67: with
68: wooden
69: writes


In [39]:
# build word -> index mapping
word_to_id = {word: idx for idx, word in enumerate(vocab)}

# build id -> word mapping
id_to_word = {idx: word for idx, word in enumerate(vocab)}

In [40]:
def text_to_token_ids(text, word_to_id):
    tokens = text.split()
    return [word_to_id[t] for t in tokens]

def token_ids_to_text(token_ids, id_to_word):
    words = [id_to_word[id] for id in token_ids]
    return " ".join(words)

In [ ]:
print(example_data)

In [41]:
text_to_token_ids(example_data, word_to_id)

[2, 11, 55, 3, 46, 45, 14, 0]

In [42]:
text_to_token_ids(example_data, word_to_id)[:CONTEXT_LEN + 1]

[2, 11, 55, 3, 46, 45, 14]

In [43]:
text_to_token_ids(example_data, word_to_id)[:CONTEXT_LEN]

[2, 11, 55, 3, 46, 45]

In [44]:
text_to_token_ids(example_data, word_to_id)[1:CONTEXT_LEN + 1]

[11, 55, 3, 46, 45, 14]

In [45]:
class LLMDataset(Dataset):
    def __init__(self, texts, word_to_id, max_len):
        self.texts = texts
        self.word_to_id = word_to_id
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = torch.tensor(text_to_token_ids(self.texts[idx], self.word_to_id)[:self.max_len + 1])
        x = tokens[:-1]
        y = tokens[1:]
        return x, y

# create dataset + dataloader
train_dataset = LLMDataset(
    text_data,
    word_to_id,
    CONTEXT_LEN
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(len(train_dataloader))

4


In [46]:
example_input_output = next(iter(train_dataloader))

In [47]:
print(example_input_output)

[tensor([[ 2, 11, 55,  3, 46, 45],
        [ 1, 49,  5,  3, 22, 60],
        [ 1, 49, 40,  3,  8, 35],
        [ 2, 11, 55,  3, 69, 41]]), tensor([[11, 55,  3, 46, 45, 14],
        [49,  5,  3, 22, 60, 38],
        [49, 40,  3,  8, 35, 18],
        [11, 55,  3, 69, 41,  0]])]


In [48]:
# Example Input
example_input_output[0][0]

tensor([ 2, 11, 55,  3, 46, 45])

In [49]:
# Example Output (shifted by one token)
example_input_output[1][0]

tensor([11, 55,  3, 46, 45, 14])

In [50]:
class GPTModel(nn.Module):
    def __init__(self):

        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMB_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LEN, EMB_DIM)
        self.output_layer = nn.Linear(EMB_DIM, VOCAB_SIZE, bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape

        tok_embeds = self.tok_emb(in_idx)

        pos_embeds = self.pos_emb(torch.arange(seq_len))

        x = tok_embeds + pos_embeds

        logits = self.output_layer(x)
        return logits

In [51]:
model = GPTModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.1)
loss_fn = nn.CrossEntropyLoss()

In [52]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {trainable_params:,}")

Total trainable parameters: 3,504


In [53]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 3,504


In [54]:
param_count = defaultdict(int)

for name, param in model.named_parameters():
    top_level = name.split('.')[0]
    param_count[top_level] += param.numel()

for k, v in param_count.items():
    print(f"{k:20s}: {v:,}")

tok_emb             : 1,680
pos_emb             : 144
output_layer        : 1,680


In [55]:
def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch, (X, y) in enumerate(dataloader):

        logits = model(X)
        loss = loss_fn(logits.flatten(0, 1), y.flatten())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(f"batch: {batch + 1} loss: {loss:>7f}")

In [56]:
for epoch in range(EPOCHS):
    print("-------------------------------")
    print(f"Epoch: {epoch+1}")
    train(train_dataloader, model, loss_fn, optimizer)
    print("-------------------------------")
print("Done!")

-------------------------------
Epoch: 1
batch: 1 loss: 4.721181
batch: 2 loss: 4.763902
batch: 3 loss: 4.540037
batch: 4 loss: 4.561274
-------------------------------
-------------------------------
Epoch: 2
batch: 1 loss: 4.688553
batch: 2 loss: 4.736847
batch: 3 loss: 4.517282
batch: 4 loss: 4.538830
-------------------------------
-------------------------------
Epoch: 3
batch: 1 loss: 4.661753
batch: 2 loss: 4.711968
batch: 3 loss: 4.494849
batch: 4 loss: 4.515833
-------------------------------
-------------------------------
Epoch: 4
batch: 1 loss: 4.635728
batch: 2 loss: 4.687508
batch: 3 loss: 4.472503
batch: 4 loss: 4.492705
-------------------------------
-------------------------------
Epoch: 5
batch: 1 loss: 4.610069
batch: 2 loss: 4.663286
batch: 3 loss: 4.450257
batch: 4 loss: 4.469583
-------------------------------
-------------------------------
Epoch: 6
batch: 1 loss: 4.584646
batch: 2 loss: 4.639246
batch: 3 loss: 4.428123
batch: 4 loss: 4.446526
------------------

In [57]:
_ = model.eval()

In [58]:
def generate(start_text):
    token_ids = text_to_token_ids(start_text, word_to_id)
    num_new_tokens = CONTEXT_LEN - len(token_ids)
    idx = torch.tensor(token_ids).unsqueeze(0)

    for _ in range(num_new_tokens):
        idx_cond = idx[:, -CONTEXT_LEN:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    idx = idx.view(-1).tolist()
    text = token_ids_to_text(idx, id_to_word)
    return text

In [59]:
text = generate("He reads")
print("Generated text:", text)

Generated text: He reads books composes and with


In [60]:
text = generate("She composes")
print("Generated text:", text)

Generated text: She composes harmonies and curates navigation
